# Validation Battery: `geotrans2` (refactored) vs `exorings`

Cross-validates the refactored `geotrans2` module (inputs: `rhotrue`, `P`, `b`, `p`)
against the analytical formulae from `exorings-basic.py`.

### Observables under test
| Observable | geotrans2 | exorings |
|---|---|---|
| Semimajor axis `a/R*` | `S.aRs` | Kepler III |
| Orbital inclination `iorb` | `S.iorb` | `arccos(b/a)` |
| Transit depth `δ` | `ringedPlanetArea(S)/π` | analytical ring area |
| Observed radius ratio `pobs/p` | `sqrt(δ)/p` | same |
| Transit durations `T14`, `T23` | `contactTimes(S)` | exorings formula |
| Photo-Ring effect `log₁₀(ρ_obs/ρ_true)` | `calculate_PR()` | exorings formula |
| Analytical ring area | `analyticalTransitAreaSystem(S)` | exorings formula |

### 12 test configurations
1. Baseline — exorings defaults  
2. Compact orbit (hot-Jupiter, P=3d)  
3. High impact parameter (b=0.70)  
4. Edge-on rings (ir=89°)  
5. Face-on rings (ir=10°)  
6. Large rings (fe=4.0 Rp)  
7. No inner ring hole (fi≈1.01)  
8. Eccentric orbit (e=0.30)  
9. Zero opacity — rings invisible  
10. Near-grazing transit  
11. Sub-Earth planet (p=0.01)  
12. Giant planet (p=0.15)  

## 0 · Setup

In [ ]:
import sys, os, warnings, importlib
import numpy as np
from numpy import pi, arccos, sqrt, cos, sin, exp, log10
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')

# ── load geotrans2 ──────────────────────────────────────
sys.path.insert(0, '.')
spec = importlib.util.spec_from_file_location('geotrans2-lite', 'geotrans2-lite.py')
gt   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gt)

RingedSystem             = gt.RingedSystem
ringedPlanetArea         = gt.ringedPlanetArea
contactTimes             = gt.contactTimes
analyticalTransitAreaSystem = gt.analyticalTransitAreaSystem
GCONST = gt.GCONST
DAY    = gt.DAY
HOUR   = gt.HOUR
DEG    = gt.DEG
RAD    = gt.RAD

RTOL = 1e-3   # 0.1 % relative tolerance
ATOL = 1e-6
print('geotrans2 loaded — validation ready.')

## 1 · Exorings reference engine

Self-contained reimplementation of the exorings analytical formulae used as ground truth.

In [ ]:
def exorings_ref(rhotrue, P, b, p, fi, fe, tau, theta_deg, ir_deg):
    # constants
    _DEG  = np.pi/180
    _RAD  = 180/np.pi
    _G    = 6.674e-11
    _DAY  = 86400.0
    _HOUR = 3600.0

    # semimajor axis a/R*
    a = (_G*(rhotrue*1e3)/(3*np.pi)*(P*_DAY)**2)**(1./3)

    cosiorb = b / a
    if abs(cosiorb) > 1:
        raise ValueError(f'No transit: b={b} > a/R*={a:.4f}')
    siniorb  = np.sqrt(1 - cosiorb**2)
    iorb_deg = np.arccos(cosiorb)*_RAD

    ir    = ir_deg*_DEG
    theta = theta_deg*_DEG
    A = fe*p
    B = A*np.cos(ir)

    hp = max(p, A*np.sin(theta), B*np.cos(theta))
    if b > 1.0 - hp:
        raise ValueError(f'Grazing/no-transit: b={b:.3f} > 1-hp={1-hp:.3f}')

    cosir = np.cos(ir); sinir = np.sin(ir)
    beta  = 1 - np.exp(-tau/cosir) if cosir > 1e-15 else 1.0

    def ring_eff(f):
        if f*cosir > 1:
            return f**2*cosir - 1
        yi = np.sqrt(f**2 - 1)/(f*sinir)
        return f**2*cosir*2/np.pi*np.arcsin(yi) - 2/np.pi*np.arcsin(yi*f*cosir)

    ri2 = beta*ring_eff(fi)
    re2 = beta*ring_eff(fe)

    ARp   = np.pi*p**2 + np.pi*(re2 - ri2)*p**2
    delta = ARp/np.pi
    pobs  = np.sqrt(delta)

    # contact positions
    xp14 = np.sqrt((1+p)**2 - b**2); xp1=-xp14; xp4=+xp14
    xp23 = np.sqrt((1-p)**2 - b**2); xp2=-xp23; xp3=+xp23

    xR13 = 1 - A**2*(np.sin(theta) - b/A)**2*(1 - B**2/A)
    xR24 = 1 - A**2*(np.sin(theta) + b/A)**2*(1 - B**2/A)
    xR1  = -np.sqrt(max(xR13,0)) - A*np.cos(theta)
    xR2  = -np.sqrt(max(xR24,0)) + A*np.cos(theta)
    xR3  = +np.sqrt(max(xR13,0)) - A*np.cos(theta)
    xR4  = +np.sqrt(max(xR24,0)) + A*np.cos(theta)
    x1 = min(xp1,xR1); x2 = max(xp2,xR2)
    x3 = min(xp3,xR3); x4 = max(xp4,xR4)

    # transit durations
    T14p = (P*_DAY)*np.arcsin((xp4-xp1)/(a*siniorb))/(2*np.pi)/_HOUR
    T23p = (P*_DAY)*np.arcsin((xp3-xp2)/(a*siniorb))/(2*np.pi)/_HOUR
    T14  = (P*_DAY)*np.arcsin((x4-x1)/(a*siniorb))/(2*np.pi)/_HOUR
    T23  = (P*_DAY)*np.arcsin((x3-x2)/(a*siniorb))/(2*np.pi)/_HOUR

    # observed density (Seager 2003)
    aobs   = 2*(P*_DAY/_HOUR)/np.pi * delta**0.25 / (T14**2 - T23**2)**0.5
    rhoobs = (3*np.pi/_G)*aobs**3/(P*_DAY)**2/1e3
    PR     = rhoobs/rhotrue
    logPR  = np.log10(PR)

    return dict(
        a=a, iorb_deg=iorb_deg, beta=beta,
        ri2=ri2, re2=re2, ARp=ARp, delta=delta, pobs=pobs,
        xp1=xp1,xp2=xp2,xp3=xp3,xp4=xp4,
        xR1=xR1,xR2=xR2,xR3=xR3,xR4=xR4,
        x1=x1,x2=x2,x3=x3,x4=x4,
        T14p=T14p,T23p=T23p,T14=T14,T23=T23,
        aobs=aobs,rhoobs=rhoobs,rhotrue=rhotrue,
        PR=PR,logPR=logPR,
    )

print('Reference engine ready.')

## 2 · Test configurations

In [ ]:
CONFIGS = [
    dict(label='01 Baseline (exorings defaults)',
         rhotrue=1.40598, P=365.2446, b=0.1875,
         p=0.08, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0),

    dict(label='02 Compact orbit (P=3d hot-Jupiter)',
         rhotrue=1.40598, P=3.0, b=0.1,
         p=0.10, fi=1.5, fe=2.5, tau=0.8,
         theta_deg=25.0, ir_deg=75.0),

    dict(label='03 High impact parameter (b=0.70)',
         rhotrue=1.40598, P=10.0, b=0.70,
         p=0.08, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0),

    dict(label='04 Edge-on rings (ir=89 deg)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.08, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=89.0),

    dict(label='05 Face-on rings (ir=10 deg)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.08, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=10.0),

    dict(label='06 Large rings (fe=4.0 Rp)',
         rhotrue=1.40598, P=100.0, b=0.10,
         p=0.06, fi=1.5, fe=4.0, tau=1.0,
         theta_deg=45.0, ir_deg=70.0),

    dict(label='07 No inner ring hole (fi=1.01)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.08, fi=1.01, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0),

    dict(label='08 Eccentric orbit (e=0.30)',
         rhotrue=1.40598, P=50.0, b=0.15,
         p=0.08, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0, ep=0.30),

    dict(label='09 Zero opacity (rings invisible)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.08, fi=1.5, fe=2.35, tau=0.0,
         theta_deg=30.0, ir_deg=80.0),

    dict(label='10 Near-grazing transit (b=0.60)',
         rhotrue=1.40598, P=30.0, b=0.60,
         p=0.08, fi=1.5, fe=2.0, tau=1.0,
         theta_deg=45.0, ir_deg=80.0),

    dict(label='11 Sub-Earth planet (p=0.01)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.01, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0),

    dict(label='12 Giant planet (p=0.15)',
         rhotrue=1.40598, P=30.0, b=0.10,
         p=0.15, fi=1.5, fe=2.35, tau=1.0,
         theta_deg=30.0, ir_deg=80.0),
]
print(f'{len(CONFIGS)} configurations defined.')

## 3 · Test runner

For each configuration we compute the exorings ground truth analytically, build a `RingedSystem` with identical parameters, and compare every observable at **0.1 % relative tolerance**.

In [ ]:
def build_system(cfg):
    ep   = cfg.get('ep', 0.0)
    phir = cfg.get('theta_deg', 30.0)*DEG
    ir   = cfg['ir_deg']*DEG
    return RingedSystem(dict(
        rhotrue=cfg['rhotrue'], P=cfg['P'], b=cfg['b'],
        p=cfg['p'], fi=cfg['fi'], fe=cfg['fe'], tau=cfg['tau'],
        phir=phir, ir=ir, ep=ep))

def run_all(configs):
    records = []
    for cfg in configs:
        label = cfg['label']
        ep    = cfg.get('ep', 0.0)
        try:
            S = build_system(cfg)
        except SystemExit:
            records.append(dict(label=label, status='SKIP (no transit)'))
            continue
        try:
            ref = exorings_ref(rhotrue=cfg['rhotrue'], P=cfg['P'],
                               b=cfg['b'], p=cfg['p'],
                               fi=cfg['fi'], fe=cfg['fe'],
                               tau=cfg['tau'],
                               theta_deg=S.teff*RAD,
                               ir_deg=S.ieff*RAD,)
                            #    theta_deg=cfg['theta_deg'],
                            #    ir_deg=cfg['ir_deg'])
        except ValueError as e:
            print(f'  SKIP {label}: {e}')
            records.append(dict(label=label, status=f'SKIP ({e})'))
            continue

        # ── observables ───────────────────────────────────────
        gt_delta  = ringedPlanetArea(S) / np.pi
        gt_pobs   = np.sqrt(gt_delta)
        gt_pobsp  = gt_pobs / cfg['p']
        ref_pobsp = ref['pobs'] / cfg['p']
        gt_Ana    = analyticalTransitAreaSystem(S) / np.pi
        ref_Ana   = ref['ARp'] / np.pi

        try:
            tcsp = contactTimes(S)
            tT   = (tcsp[-1]-tcsp[1])/HOUR
            tF   = (tcsp[-2]-tcsp[2])/HOUR
        except Exception:
            tT = tF = np.nan

        try:
            logPR_gt = S.calculate_PR()
        except Exception:
            logPR_gt = np.nan

        # ── comparison helper ─────────────────────────────────
        def close(a, b):
            return abs(a-b)/max(abs(b), ATOL) < RTOL, abs(a-b)/max(abs(b),ATOL)*100

        checks = {}
        ok, e = close(S.aRs, ref['a']); checks['a/R* (%)']   = (ok, e)
        ok, e = close(np.degrees(S.iorb), ref['iorb_deg']); checks['iorb (%)'] = (ok, e)
        ok, e = close(gt_delta, ref['delta']); checks['delta (%)']  = (ok, e)
        ok, e = close(gt_pobsp, ref_pobsp);   checks['pobs/p (%)']  = (ok, e)
        if ep == 0.0:
            ok, e = close(gt_Ana, ref_Ana);   checks['Anal.Area (%)'] = (ok, e)
        if ep == 0.0 and not np.isnan(tT):
            ok, e = close(tT, ref['T14']);     checks['T14 (%)']   = (ok, e)
            ok, e = close(tF, ref['T23']);     checks['T23 (%)']   = (ok, e)
        if not np.isnan(logPR_gt):
            ok, e = close(logPR_gt, ref['logPR']); checks['logPR (%)'] = (ok, e)

        n_pass = sum(v[0] for v in checks.values())
        n_tot  = len(checks)
        status = 'PASS' if n_pass == n_tot else f'PARTIAL ({n_pass}/{n_tot})'

        row = dict(label=label, status=status,
                   aRs_gt=S.aRs, aRs_ref=ref['a'],
                   delta_ppm_gt=gt_delta*1e6, delta_ppm_ref=ref['delta']*1e6,
                   pobs_p_gt=gt_pobsp, pobs_p_ref=ref_pobsp,
                   T14_gt=tT, T14_ref=ref['T14'],
                   T23_gt=tF, T23_ref=ref['T23'],
                   logPR_gt=logPR_gt, logPR_ref=ref['logPR'],
                   **{k: v[1] for k,v in checks.items()})
        records.append(row)
        sym = chr(10003) if status=='PASS' else chr(10007)
        print(f'  {sym}  [{status}]  {label}')
    return records

print('Running validation battery...\n')
records = run_all(CONFIGS)

## 4 · Summary table

In [ ]:
df = pd.DataFrame(records)

def highlight(v):
    if isinstance(v, str):
        if v == 'PASS':     return 'background-color:#d4edda;color:#155724;font-weight:bold'
        if 'PARTIAL' in v:  return 'background-color:#fff3cd;color:#856404;font-weight:bold'
        if 'SKIP'    in v:  return 'background-color:#cce5ff;color:#004085'
    if isinstance(v, float) and v > 0.1:
        return 'background-color:#f8d7da'
    return ''

err_cols = [c for c in df.columns if c.endswith(' (%)')]
show_cols = ['label','status'] + err_cols
show_cols = [c for c in show_cols if c in df.columns]
fmt = {c: '{:.4f}' for c in err_cols if c in df.columns}

styled = (df[show_cols].fillna('—')
          .style
          .applymap(highlight))
        #   .format(fmt, na_rep='—'))
display(styled)

## 5 · Side-by-side numeric comparison

In [ ]:
pairs = [
    ('a/R*',    'aRs_gt',       'aRs_ref',      '{:.4f}'),
    ('d (ppm)', 'delta_ppm_gt', 'delta_ppm_ref', '{:.3f}'),
    ('pobs/p',  'pobs_p_gt',    'pobs_p_ref',    '{:.5f}'),
    ('T14 (h)', 'T14_gt',       'T14_ref',       '{:.5f}'),
    ('T23 (h)', 'T23_gt',       'T23_ref',       '{:.5f}'),
    ('logPR',   'logPR_gt',     'logPR_ref',     '{:.5f}'),
]

rows2 = []
for rec in records:
    if 'aRs_gt' not in rec: continue
    for name, kg, kr, fmt in pairs:
        vg = rec.get(kg, np.nan); vr = rec.get(kr, np.nan)
        if np.isnan(vg) or np.isnan(vr): continue
        rel = abs(vg-vr)/max(abs(vr),1e-6)*100
        rows2.append(dict(
            config=rec['label'][:35],
            observable=name,
            geotrans2=fmt.format(vg),
            exorings=fmt.format(vr),
            rel_err_pct=f'{rel:.4f}'))

df2 = pd.DataFrame(rows2)

def color_rel(v):
    try:
        fv = float(v)
        if fv < 0.01:  return 'color:#155724'
        if fv < 0.1:   return 'color:#856404'
        return 'background-color:#f8d7da;color:#721c24'
    except: return ''

styled2 = (df2.set_index(['config','observable'])
           .style
           .applymap(color_rel, subset=['rel_err_pct']))
display(styled2)

## 6 · Relative error heatmap

Green < 0.01 % · Yellow 0.01–0.1 % · Red > 0.1 %

In [ ]:
err_cols  = [c for c in df.columns if c.endswith(' (%)')]
valid_recs = [r for r in records if 'aRs_gt' in r]
labels_h   = [r['label'][:40] for r in valid_recs]
M = np.array([[r.get(c, np.nan) for c in err_cols] for r in valid_recs], dtype=float)

fig, ax = plt.subplots(figsize=(13, max(4, len(labels_h)*0.6)))
im = ax.imshow(M, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.10)
plt.colorbar(im, ax=ax, label='Relative error (%)')
ax.set_xticks(range(len(err_cols)))
ax.set_xticklabels([c.replace(' (%)','') for c in err_cols],
                   rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(labels_h)))
ax.set_yticklabels(labels_h, fontsize=9)
ax.set_title('geotrans2 vs exorings — relative error (%) | Green<0.01% Yellow<0.1% Red>0.1%')
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i,j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    fontsize=7.5, color='black' if v < 0.05 else 'white')
plt.tight_layout()
plt.savefig('validation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Transit depth comparison

In [ ]:
gt_d  = [r['delta_ppm_gt']  for r in records if 'delta_ppm_gt'  in r]
ref_d = [r['delta_ppm_ref'] for r in records if 'delta_ppm_ref' in r]
lbls  = [r['label'][:28]    for r in records if 'delta_ppm_gt'  in r]
rel_e = [abs(g-r)/max(abs(r),1e-6)*100 for g,r in zip(gt_d,ref_d)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(lbls)); w = 0.38

ax = axes[0]
ax.bar(x-w/2, gt_d,  w, label='geotrans2', color='#2196F3', alpha=0.85)
ax.bar(x+w/2, ref_d, w, label='exorings',  color='#FF9800', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Transit depth (ppm)'); ax.set_title('Transit depth'); ax.legend()

ax = axes[1]
colors = ['#4CAF50' if e<0.01 else '#FF9800' if e<0.1 else '#F44336' for e in rel_e]
ax.barh(lbls, rel_e, color=colors)
ax.axvline(0.10,  color='orange', ls='--', lw=1, label='0.10 %')
ax.axvline(0.01,  color='green',  ls='--', lw=1, label='0.01 %')
ax.set_xlabel('Relative error (%)'); ax.set_title('Relative error in delta'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('validation_delta.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Transit durations (T14, T23) — 1:1 scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (kg, kr, title, col) in zip(axes, [
    ('T14_gt','T14_ref','T14 total duration','#3F51B5'),
    ('T23_gt','T23_ref','T23 flat-bottom','#9C27B0')]):
    gts=[]; refs=[]; lbls=[]
    for r in records:
        g=r.get(kg,np.nan); rv=r.get(kr,np.nan)
        if not(np.isnan(g) or np.isnan(rv) or 'ep' in str(r.get('label','') and r.get('ep',0)!=0)):
            gts.append(g); refs.append(rv)
            lbls.append(r['label'][:20])
    if not gts: ax.set_title(f'{title} (no data)'); continue
    mn=min(min(gts),min(refs))*0.9; mx=max(max(gts),max(refs))*1.1
    ax.plot([mn,mx],[mn,mx],'k--',lw=1,label='1:1')
    ax.scatter(refs, gts, c=col, s=60, zorder=3)
    for g,rv,l in zip(gts,refs,lbls):
        ax.annotate(l, (rv,g), fontsize=6, xytext=(3,3), textcoords='offset points')
    rels=[abs(g-rv)/max(abs(rv),1e-6)*100 for g,rv in zip(gts,refs)]
    ax.set_xlabel(f'exorings {title} (h)')
    ax.set_ylabel(f'geotrans2 {title} (h)')
    ax.set_title(f'{title}  max err={max(rels):.4f}%')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('validation_durations.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Photo-Ring effect

In [ ]:
gts  = [r.get('logPR_gt',  np.nan) for r in records if 'aRs_gt' in r]
refs = [r.get('logPR_ref', np.nan) for r in records if 'aRs_gt' in r]
lbls = [r['label'][:30]             for r in records if 'aRs_gt' in r]
x = np.arange(len(lbls)); w = 0.38

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(x-w/2, gts,  w, label='geotrans2', color='#0277BD', alpha=0.85)
ax.bar(x+w/2, refs, w, label='exorings',  color='#F57C00', alpha=0.85)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('log10(rho_obs / rho_true)')
ax.set_title('Photo-Ring Effect: geotrans2 vs exorings')
ax.legend()
for xi,(g,rv) in enumerate(zip(gts,refs)):
    if not(np.isnan(g) or np.isnan(rv)):
        rel=abs(g-rv)/max(abs(rv),1e-6)*100
        ax.text(xi, max(g,rv)+0.005, f'{rel:.3f}%',
                ha='center', fontsize=6.5,
                color='#4CAF50' if rel<0.1 else '#F44336')
plt.tight_layout()
plt.savefig('validation_PR.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Edge case — Zero opacity (tau = 0)

With `tau=0`, rings are invisible. Transit depth must equal `p²` exactly.

In [ ]:
cfg0 = next(c for c in CONFIGS if 'Zero' in c['label'])
S0   = build_system(cfg0)
d_gt = ringedPlanetArea(S0) / np.pi
d_exp = cfg0['p']**2
rel = abs(d_gt - d_exp)/d_exp*100
print(f'tau=0 test')
print(f'  Expected p^2   = {d_exp:.10f}')
print(f'  geotrans2 delta= {d_gt:.10f}')
print(f'  Absolute diff  = {abs(d_gt-d_exp):.2e}')
print(f'  Relative error = {rel:.6f} %')
assert rel < RTOL*100, f'FAIL: tau=0 depth != p^2 (err={rel:.4f}%)'
print('  PASS: delta == p^2 within tolerance')

## 11 · Sweep — Ring inclination (ir: 10° → 89°)

At ir=89° (edge-on) the projected ring area → 0 and depth → naked-planet value.

In [ ]:
p_t = 0.08
ir_degs = [5,10,20,30,40,50,60,70,80,85,89]
d_gts=[]; d_refs=[]
for ir_d in ir_degs:
    cfg={'rhotrue':1.40598,'P':30.0,'b':0.10,'p':p_t,
         'fi':1.5,'fe':2.35,'tau':1.0,'theta_deg':30.0,'ir_deg':ir_d}
    try:
        S=build_system(cfg)
        cfg['theta_deg']=S.teff*RAD
        cfg['ir_deg']=S.ieff*RAD
        ref=exorings_ref(**cfg)
        d_gts.append(ringedPlanetArea(S)/np.pi)
        d_refs.append(ref['delta'])
    except: d_gts.append(np.nan); d_refs.append(np.nan)

fig,ax=plt.subplots(figsize=(9,4))
ax.plot(ir_degs,[d*1e6 for d in d_gts], 'o-', label='geotrans2', lw=2)
ax.plot(ir_degs,[d*1e6 for d in d_refs],'s--', label='exorings',  lw=2)
ax.axhline(p_t**2*1e6, color='gray', ls=':', label=f'naked planet p^2={p_t**2*1e6:.0f} ppm')
ax.set_xlabel('Ring inclination (deg)'); ax.set_ylabel('Transit depth (ppm)')
ax.set_title('Depth vs ring inclination'); ax.legend()
plt.tight_layout()
plt.savefig('validation_ir_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print('ir  | gt (ppm)  | ref (ppm) | err (%)')
for ir_d,dg,dr in zip(ir_degs,d_gts,d_refs):
    if not np.isnan(dg):
        rel=abs(dg-dr)/max(abs(dr),1e-6)*100
        print(f'{ir_d:3d}  {dg*1e6:10.3f}   {dr*1e6:10.3f}   {rel:.5f}')

## 12 · Sweep — Ring opacity (tau: 0 → 2)

Depth saturates at high opacity; relative error must stay < 0.1 % throughout.

In [ ]:
taus = np.linspace(0, 2.0, 30)
base = dict(rhotrue=1.40598,P=30.0,b=0.10,p=0.08,fi=1.5,fe=2.35,
            theta_deg=30.0,ir_deg=80.0)
d_gts2=[]; d_refs2=[]
for tau in taus:
    cfg={**base,'tau':tau}
    try:
        S=build_system(cfg)
        cfg['theta_deg']=S.teff*RAD
        cfg['ir_deg']=S.ieff*RAD
        ref=exorings_ref(**cfg)
        d_gts2.append(ringedPlanetArea(S)/np.pi); d_refs2.append(ref['delta'])
    except: d_gts2.append(np.nan); d_refs2.append(np.nan)

rels2=[abs(g-r)/max(abs(r),1e-6)*100
       for g,r in zip(d_gts2,d_refs2) if not(np.isnan(g) or np.isnan(r))]
ts2  =[t for t,g,r in zip(taus,d_gts2,d_refs2) if not(np.isnan(g) or np.isnan(r))]

fig,axes=plt.subplots(1,2,figsize=(13,4))
ax=axes[0]
ax.plot(taus,[d*1e6 for d in d_gts2], '-',  label='geotrans2',lw=2)
ax.plot(taus,[d*1e6 for d in d_refs2],'--', label='exorings', lw=2)
ax.axhline(0.08**2*1e6,color='gray',ls=':',label='naked planet')
ax.set_xlabel('tau'); ax.set_ylabel('delta (ppm)'); ax.set_title('Depth vs opacity'); ax.legend()
ax=axes[1]
ax.semilogy(ts2,rels2,'o-',color='#E91E63',ms=4)
ax.axhline(0.1,color='orange',ls='--',label='0.1 %')
ax.set_xlabel('tau'); ax.set_ylabel('Relative error (%)'); ax.set_title('Error vs opacity'); ax.legend()
plt.tight_layout()
plt.savefig('validation_tau_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Max relative error across opacity sweep: {max(rels2):.5f} %')

## 13 · Sweep — Orbital period (P: 0.3 d → 3000 d)

Validates Kepler-III inversion of `a/R*` over four decades of period.

In [ ]:
periods = np.logspace(-0.5, 3.5, 35)
base3=dict(rhotrue=1.40598,b=0.10,p=0.08,fi=1.5,fe=2.35,tau=1.0,
           theta_deg=30.0,ir_deg=80.0)
aRs_gts=[]; aRs_refs=[]
for P in periods:
    cfg={**base3,'P':float(P)}
    try:
        S=build_system(cfg)
        cfg['theta_deg']=S.teff*RAD
        cfg['ir_deg']=S.ieff*RAD
        ref=exorings_ref(**cfg)
        aRs_gts.append(S.aRs); aRs_refs.append(ref['a'])
    except: aRs_gts.append(np.nan); aRs_refs.append(np.nan)

rels3=[abs(g-r)/max(abs(r),1e-6)*100
       for g,r in zip(aRs_gts,aRs_refs) if not(np.isnan(g) or np.isnan(r))]
ps3  =[p for p,g,r in zip(periods,aRs_gts,aRs_refs) if not(np.isnan(g) or np.isnan(r))]

fig,axes=plt.subplots(1,2,figsize=(13,4))
ax=axes[0]
ax.loglog(periods,aRs_refs,'s--',ms=4,label='exorings',alpha=0.7)
ax.loglog(periods,aRs_gts, 'o-', ms=4,label='geotrans2',alpha=0.7)
ax.set_xlabel('P (days)'); ax.set_ylabel('a/R*'); ax.set_title('a/R* vs P'); ax.legend()
ax=axes[1]
ax.semilogx(ps3,rels3,'o-',color='#673AB7',ms=4)
ax.axhline(0.001,color='green',ls='--',label='0.001 %')
ax.set_xlabel('P (days)'); ax.set_ylabel('Rel. error in a/R* (%)'); ax.legend()
ax.set_title('a/R* precision across period range')
plt.tight_layout()
plt.savefig('validation_period_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Max relative error in a/R* across period sweep: {max(rels3):.2e} %')

## 14 · Eccentric orbit self-consistency (e = 0 → 0.5)

Transit depth depends only on ring geometry (not orbital shape). Verifies that `rcen/aRs` and `Borb` remain self-consistent for varying eccentricity.

In [ ]:
print('e     a/R*       rcen/a     |Borb-b|/b (%)   delta (ppm)')
for ep in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    cfg=dict(rhotrue=1.40598,P=50.0,b=0.15,p=0.08,
             fi=1.5,fe=2.35,tau=1.0,theta_deg=30.0,ir_deg=80.0,ep=ep)
    try:
        S=build_system(cfg)
        d=ringedPlanetArea(S)/np.pi
        b_err=abs(abs(S.Borb)-0.15)/0.15*100
        print(f'{ep:.1f}  {S.aRs:9.4f}  {S.rcen/S.aRs:.6f}  {b_err:12.6f}         {d*1e6:.3f}')
    except SystemExit:
        print(f'{ep:.1f}  -> no transit')

## 15 · Light curve preview — Baseline configuration

Full limb-darkened transit computed with `fluxLimbTime`. The flat bottom should reach `1 - δ` from both geotrans2 and exorings.

In [ ]:
cfg_b = CONFIGS[0]
Sb = build_system(cfg_b)
Sb.c1 = 0.70; Sb.c2 = -0.24
Ar_b = ringedPlanetArea(Sb)

tcsp = contactTimes(Sb)
t1,t4 = tcsp[1], tcsp[-1]
pad = (t4-t1)*0.18
t_arr = np.linspace(t1-pad, t4+pad, 130)
flux  = [gt.fluxLimbTime(t, Ar_b, Sb) for t in t_arr]

d_b   = Ar_b/np.pi
ref_b = exorings_ref(**{k:v for k,v in cfg_b.items() if k!='label'})

fig,ax = plt.subplots(figsize=(12,4.5))
t_h = (t_arr - Sb.tcen)/HOUR
ax.plot(t_h, flux, '-', lw=2, color='#1565C0', label='geotrans2 (limb-darkened)')
ax.axhline(1-d_b,       color='#E53935', ls='--', lw=1.5,
           label=f'geotrans2 1-delta = {1-d_b:.6f}')
ax.axhline(1-ref_b['delta'], color='#43A047', ls=':', lw=1.5,
           label=f'exorings  1-delta = {1-ref_b["delta"]:.6f}')
for tc,lab in zip(tcsp[1:],['t1','t2','t3','t4']):
    ax.axvline((tc-Sb.tcen)/HOUR, color='gray', ls=':', lw=0.8)
    ax.text((tc-Sb.tcen)/HOUR, min(flux)*0.9998, lab,
            ha='center', fontsize=9, color='gray')
ax.set_xlabel('Time from mid-transit (h)')
ax.set_ylabel('Normalised flux')
ax.set_title('Light curve — Baseline configuration')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('validation_lightcurve.png', dpi=150, bbox_inches='tight')
plt.show()

T14_gt = (tcsp[-1]-tcsp[1])/HOUR
T23_gt = (tcsp[-2]-tcsp[2])/HOUR
print(f'T14  geotrans2={T14_gt:.5f} h   exorings={ref_b["T14"]:.5f} h  '
      f'err={abs(T14_gt-ref_b["T14"])/ref_b["T14"]*100:.4f}%')
print(f'T23  geotrans2={T23_gt:.5f} h   exorings={ref_b["T23"]:.5f} h  '
      f'err={abs(T23_gt-ref_b["T23"])/ref_b["T23"]*100:.4f}%')

## 16 · Final validation report

In [ ]:
valid = [r for r in records if 'aRs_gt' in r]
pass_n  = sum(1 for r in valid if r.get('status')=='PASS')
total_n = len(valid)
skip_n  = sum(1 for r in records if 'SKIP' in r.get('status',''))
all_errs = [v for r in valid for k,v in r.items()
            if k.endswith(' (%)') and isinstance(v,float) and not np.isnan(v)]
print('='*60)
print('VALIDATION SUMMARY')
print('='*60)
print(f'  Configurations tested : {total_n}')
print(f'  Skipped               : {skip_n}')
print(f'  PASS                  : {pass_n} / {total_n}')
print(f'  Max relative error    : {max(all_errs):.5f} %')
print(f'  Mean relative error   : {np.mean(all_errs):.5f} %')
print(f'  Tolerance applied     : {RTOL*100:.1f} %')
print('='*60)
if pass_n == total_n:
    print('  ALL TESTS PASSED')
else:
    failed = [r['label'] for r in valid if r.get('status','')!='PASS']
    print(f'  FAILED: {failed}')
print('='*60)